# 05: Custom Dataset Training & Inference (Roboflow Integration)
Interactive notebook to prepare a custom dataset, train **tinyYOLO-det** using transfer learning (pre-trained ImageNet GhostNet backbone), plot training curves, and validate predictions on custom test images.

Designed to run in local Jupyter Lab, Google Colab, or Kaggle.

In [ ]:
# %% Setup & Environment Installation
import os, sys, socket
try:
    socket.create_connection(("github.com", 80), timeout=5)
except:
    print("❌ ERROR: No Internet connection. If on Kaggle, toggle 'Internet' to ON in the sidebar settings.")
    sys.exit(1)

# Clone repository if running in a cloud VM (Colab / Kaggle)
if not os.path.exists('scripts/train.py'):
    print("🚀 Cloud Environment Detected. Cloning repository...")
    !rm -rf tinyYOLO && git clone https://github.com/ShMazumder/tinyYOLO.git
    %cd tinyYOLO
else:
    print("✓ Local Workspace Detected. Running in project root.")

# Install dependencies in editable mode
!pip install -e . -q
!pip install tqdm timm ultralytics matplotlib pyyaml opencv-python -q

## 0. Download Custom Dataset from Roboflow
Run this cell to install the Roboflow library, authenticate, and download your crime detection dataset in YOLO format.

In [ ]:
# %% Download from Roboflow
!pip install roboflow
from roboflow import Roboflow

# Define your Roboflow Private API Key here (or retrieve it from environment)
RB_KEY = "YOUR_ROBOFLOW_API_KEY_HERE"

rf = Roboflow(api_key=RB_KEY)
project = rf.workspace("shmazumder").project("crime-detection-oodyj-4mgzl")
version = project.version(1)
dataset = version.download("yolo26")

## 1. Parse Downloaded Dataset & Set DATASET_NAME
Roboflow automatically downloads, extracts, and organizes the image/label folders and generates a custom `data.yaml` configuration. Run this cell to read the path dynamically.

In [ ]:
# %% Parse Roboflow dataset information
import yaml
from pathlib import Path

# Roboflow automatically saves the absolute download location in dataset.location
dataset_dir = Path(dataset.location)
DATASET_NAME = dataset_dir.name  # Typically 'crime-detection-1'
roboflow_yaml_path = dataset_dir / "data.yaml"

print(f"✓ Dataset Download Path: {dataset_dir}")
print(f"✓ DATASET_NAME resolved to: {DATASET_NAME}")

# Verify directories inside downloaded folder
print("Roboflow directory contents:")
for p in dataset_dir.glob("*"):
    if p.is_dir():
        print(f"  ├── {p.name}/ (contains {len(list(p.glob('*')))} files)")

# Load class names directly from Roboflow configuration file
if roboflow_yaml_path.exists():
    with open(roboflow_yaml_path) as f:
        roboflow_config = yaml.safe_load(f)
    
    nc = roboflow_config.get("nc", 0)
    names = roboflow_config.get("names", {})
    print(f"\n✓ Loaded config from Roboflow 'data.yaml':")
    print(f"  - Classes count (nc): {nc}")
    print(f"  - Class mapping (names): {names}")
else:
    print(f"⚠️ Warning: data.yaml not found at {roboflow_yaml_path}. Verify directory contents.")

## 2. Train on Custom Dataset (Transfer Learning)
Run the training loop. We point the training script directly to the Roboflow-generated `data.yaml` config path, bypassing any manual setup. We pass `--pretrained` to use our ImageNet GhostNet weights.

In [ ]:
# %% Train tinyYOLO-det standard variant
VARIANT = "standard"  # standard | quantized
IMGSZ = 320           # Input resolution
EPOCHS = 50           # Epoch count
BATCH = 32            # Batch size (set to -1 to auto-detect optimal size)
EXP_NAME = "crime-detection-yolo-run"

# Use the Roboflow-provided data.yaml config path
data_yaml_arg = str(roboflow_yaml_path)

!python3 scripts/train.py \
    --task det \
    --variant {VARIANT} \
    --data {data_yaml_arg} \
    --imgsz {IMGSZ} \
    --epochs {EPOCHS} \
    --batch {BATCH} \
    --pretrained \
    --compile \
    --name {EXP_NAME}

## 3. Visualize Metrics & Loss Curves
Load the training `history.json` and plot Box Loss, Cls Loss, and validation mAP curves.

In [ ]:
# %% Plot training history
import json
import matplotlib.pyplot as plt

runs_dir = Path("experiments/results")
run_folders = sorted(runs_dir.glob(f"*{EXP_NAME}*"))

if run_folders:
    latest_run = run_folders[-1]
    history_file = latest_run / "history.json"
    
    if history_file.exists():
        with open(history_file) as f:
            history = json.load(f)
            
        epochs = [x['epoch'] for x in history]
        box_loss = [x['loss_box'] for x in history]
        cls_loss = [x['loss_cls'] for x in history]
        total_loss = [x['loss_total'] for x in history]
        map50 = [x.get('mAP50', 0) for x in history]
        
        # Plotting
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # Losses
        ax1.plot(epochs, total_loss, label='Total Loss', color='#1f77b4', linewidth=2)
        ax1.plot(epochs, box_loss, label='Box Loss (CIoU)', color='#ff7f0e', linestyle='--')
        ax1.plot(epochs, cls_loss, label='Cls Loss (BCE)', color='#2ca02c', linestyle=':')
        ax1.set_title("Training Losses")
        ax1.set_xlabel("Epoch")
        ax1.set_ylabel("Loss")
        ax1.grid(True, alpha=0.3)
        ax1.legend()
        
        # mAP Accuracy
        ax2.plot(epochs, map50, label='mAP@50', color='#9467bd', linewidth=2)
        ax2.set_title("Validation mAP@50")
        ax2.set_xlabel("Epoch")
        ax2.set_ylabel("mAP")
        ax2.grid(True, alpha=0.3)
        ax2.legend()
        
        plt.tight_layout()
        plt.show()
    else:
        print(f"history.json not found in {latest_run}")
else:
    print(f"No run folders found matching name: {EXP_NAME}")

## 4. Bounding Box Inference
Load your trained model weights (`best.pt`), perform NMS post-processing, and draw bounding boxes on a custom image.

In [ ]:
# %% Bounding box inference helper
import cv2
import torch
from tinyYOLO.models import build_model
from tinyYOLO.utils.postprocess import postprocess_detections

def run_custom_inference(image_path, weights_path, nc, class_names, imgsz=320):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Build model & load state dict
    model, _ = build_model(task='det', variant=VARIANT, nc=nc)
    checkpoint = torch.load(weights_path, map_location=device)
    state_dict = checkpoint['model'] if 'model' in checkpoint else checkpoint
    model.load_state_dict(state_dict)
    model.to(device).eval()
    
    # Preprocess image
    img0 = cv2.imread(str(image_path))
    h0, w0 = img0.shape[:2]
    img_rgb = cv2.cvtColor(img0, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (imgsz, imgsz))
    
    img_tensor = torch.from_numpy(img_resized.transpose(2, 0, 1)).float() / 255.0
    img_tensor = img_tensor.unsqueeze(0).to(device)
    
    with torch.no_grad():
        preds = model(img_tensor)
        
    # Decode predictions & apply NMS
    detections = postprocess_detections(preds, conf_thres=0.25, iou_thres=0.45)[0]
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(img_rgb)
    
    if len(detections) > 0:
        print(f"✓ Detected {len(detections)} objects:")
        for det in detections:
            x1, y1, x2, y2 = int(det[0]*w0), int(det[1]*h0), int(det[2]*w0), int(det[3]*h0)
            score = det[4]
            cls_id = int(det[5])
            
            name = class_names[cls_id] if class_names and cls_id in class_names else f"cls_{cls_id}"
            print(f"  - {name}: {score*100:.1f}%")
            
            rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color='#ff0000', linewidth=3)
            ax.add_patch(rect)
            ax.text(x1, y1-10, f"{name} {score:.2f}", color='white', fontsize=12,
                    bbox=dict(facecolor='#ff0000', alpha=0.8, edgecolor='none', boxstyle='round,pad=0.2'))
    else:
        print("No objects detected.")
    
    plt.axis('off')
    plt.show()

# To test inference, uncomment and run:
# ======================================
# if run_folders:
#     # Get a sample image from val directory dynamically
#     val_images = list(dataset_dir.glob("val/images/*"))
#     if val_images:
#         run_custom_inference(
#             image_path=val_images[0],
#             weights_path=run_folders[-1] / "best.pt",
#             nc=nc,
#             class_names=names
#         )
#     else:
#         print("No validation images found in datasets folder.")
